In [5]:
# ContextualCompressionRetriever
!pip install langchain-classic
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings,ChatHuggingFace,HuggingFaceEndpoint
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
# pip install langchain-classic


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Recreate the document objects from the previous data
docs=[
    Document(
        page_content="During the medieval era, the concept of tourism as leisure travel was virtually nonexistent compared to modern standards. Photosynthesis is process of plants amking food using chlorophyll and sunlight.Travel in the Middle Ages was primarily motivated by necessity rather than pleasure. Most journeys were undertaken for religious, economic, or political reasons rather than for recreation or sightseeing.",id="mixed_data",
        metadata={"source":"doc1"}
    )
]

In [8]:
embedding_model=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1262.88it/s]


#create a FAISS store from the documents

In [ ]:
vectorstore=FAISS.from_documents(docs,embedding_model)
vectorstore.save_local("faiss_index1")

In [10]:
base_retriever=vectorstore.as_retriever(search_kwargs={"k":1})

In [20]:
# Setup the compressor using an LLM
from dotenv import load_dotenv 
load_dotenv()
llm=HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",task="text-generation",max_new_tokens=30
)
llm=ChatHuggingFace(llm=llm)
compressor=LLMChainExtractor.from_llm(llm)

In [21]:
# Create the contextual compression retriever
compression_retriever=ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [22]:
# Query the retriever
query="What is Photosynthesis in 1 line?"
compressed_results=compression_retriever.invoke(query)

In [23]:
compressed_results

[Document(metadata={'source': 'doc1'}, page_content='> Photosynthesis is process of plants amking food using chlorophyll and sunlight.')]

In [24]:
for i,doc in enumerate(compressed_results):
    print(f"\n ----Result{i+1}---")
    print(doc.page_content)


 ----Result1---
> Photosynthesis is process of plants amking food using chlorophyll and sunlight.
